In [0]:
# Databricks notebook source
# COMMAND ----------

# DBTITLE 1,Paths
BASE_PATH = "abfss://atininput@sadbtrng19032026.dfs.core.windows.net/data/"
STUDENT_ID = "u25"  # Change to your assigned u01–u16
DAY5 = BASE_PATH + "/day5"
P_BASIC = DAY5 + "/delta/flight_summary_basic"
DAY6_PREFIX = f"{BASE_PATH}day06-{STUDENT_ID}"

# COMMAND ----------

# DBTITLE 1,Prerequisite Check
try:
    spark.read.format("delta").load(P_BASIC).limit(1).collect()
    print("✓ Day 5 tables found (P_BASIC)")
except Exception as e:
    print(f"✗ Day 5 tables not found: {e}")
    print("Please complete Day 5 notebook 01 first.")

# COMMAND ----------


# Time travel and table history

Covers Delta history, `DESCRIBE HISTORY`, and reading older data with `versionAsOf` / `timestampAsOf` (or SQL `VERSION AS OF` / `TIMESTAMP AS OF`).

Lab 1: write data, change it, inspect history, read an older version.  
Lab 2: restore.  
Lab 3: MERGE from a second Delta path (batch upsert; same line of work as Day 5).

Streaming and change data feed are on Day 7.

---

## Lab 1
### Step 1: Create a Delta table


In [0]:
# Default Spark session on the cluster
data = [(101, 5000, "2024-02-01"), (102, 7000, "2024-02-02"), (103, 4500, "2024-02-03")]
columns = ["transaction_id", "amount", "transaction_date"]

df = spark.createDataFrame(data, columns)
df.write.format("delta").mode("overwrite").save(f"{DAY6_PREFIX}/transactions")


### Step 2: Update and view history

In [0]:
from delta.tables import DeltaTable
from pyspark.sql.functions import lit

delta_table = DeltaTable.forPath(spark, f"{DAY6_PREFIX}/transactions")

delta_table.update("transaction_id = 101", {"amount": lit(6000)})

spark.sql(f"DESCRIBE HISTORY delta.`{DAY6_PREFIX}/transactions`").show(20, truncate=False)


### Step 3: Read an older version

In [0]:
previous_version = 0  # Adjust based on the output of history()
delta_df = spark.read.format("delta").option("versionAsOf", previous_version).load(f"{DAY6_PREFIX}/transactions")
delta_df.show()



---

## Lab 2: Restore

### Step 1: Restore to a version

In [0]:
# Version 0 is the first commit; version 1 is after the update. Adjust from DESCRIBE HISTORY if needed.
restore_version = 0
delta_table = DeltaTable.forPath(spark, f"{DAY6_PREFIX}/transactions")
delta_table.restoreToVersion(restore_version)


### Step 2: Read the table

In [0]:
spark.read.format("delta").load(f"{DAY6_PREFIX}/transactions").show()



---

## Lab 3: MERGE

### Step 1: Write a source Delta path

In [0]:
updated_data = [(101, 5500, "2024-02-01"), (104, 9000, "2024-02-05")]  # New transaction
source_df = spark.createDataFrame(updated_data, columns)
source_df.write.format("delta").mode("overwrite").save(f"{DAY6_PREFIX}/updates")



### Step 2: MERGE into the target

In [0]:
from pyspark.sql.functions import col
from delta.tables import DeltaTable

delta_table = DeltaTable.forPath(spark, f"{DAY6_PREFIX}/transactions")
source_table = DeltaTable.forPath(spark, f"{DAY6_PREFIX}/updates")

delta_table.alias("t").merge(
    source_table.toDF().alias("s"),
    "t.transaction_id = s.transaction_id",
).whenMatchedUpdate(
    set={"amount": col("s.amount"), "transaction_date": col("s.transaction_date")}
).whenNotMatchedInsert(
    values={
        "transaction_id": col("s.transaction_id"),
        "amount": col("s.amount"),
        "transaction_date": col("s.transaction_date"),
    }
).execute()


### Step 3: Check the result

In [0]:
from delta.tables import DeltaTable

DeltaTable.forPath(spark, f"{DAY6_PREFIX}/transactions").toDF().show()


## Note (Day 7)

Change data feed and streaming are covered later. For a batch-only preview, see the Databricks docs for `readChangeFeed` / `table_changes` on your Delta path.
